In [10]:
import pandas as pd

In [11]:
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [12]:
df = pd.read_csv('../data/cicids2017_cleaned.csv')

In [13]:
print(f"Shape: {df.shape}")

Shape: (691406, 79)


In [14]:
print(df.columns.tolist())

[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count', ' URG Flag 

In [15]:
print(f"\nLabel distribution:\n{df[' Label'].value_counts()}")


Label distribution:
 Label
BENIGN              439683
DoS Hulk            230124
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64


In [16]:
df['is_attack'] = (df[' Label'] != 'BENIGN').astype(int)

In [17]:
label_encoder = LabelEncoder()

In [18]:
df['attack_type_encoded'] = label_encoder.fit_transform(df[' Label'])


In [19]:
import pickle

In [22]:
import os
os.makedirs('../results', exist_ok=True)

In [23]:
pickle.dump(label_encoder, open('../results/label_encoder.pkl', 'wb'))

In [24]:
print("Label classes:", label_encoder.classes_)
print(f"\nBinary distribution:\n{df['is_attack'].value_counts()}")

Label classes: ['BENIGN' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest' 'DoS slowloris'
 'Heartbleed']

Binary distribution:
is_attack
0    439683
1    251723
Name: count, dtype: int64


In [26]:
X = df.drop([' Label', 'is_attack', 'attack_type_encoded'], axis=1)
y = df['is_attack']

print(f"Features shape: {X.shape}")
print(f"Label shape: {y.shape}")

Features shape: (691406, 78)
Label shape: (691406,)


In [27]:
from sklearn.preprocessing import StandardScaler

In [28]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [29]:
pickle.dump(scaler, open('../results/scaler.pkl', 'wb'))

In [30]:
print("Scaler saved.")

Scaler saved.


In [32]:
#time to train-test split the data
from sklearn.model_selection import train_test_split

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [34]:
os.makedirs('../data', exist_ok=True)

In [35]:
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
pd.DataFrame({'y': y_train}).to_csv('../data/y_train.csv', index=False)
pd.DataFrame({'y': y_test}).to_csv('../data/y_test.csv', index=False)


In [36]:
print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")
print("All splits saved!")

Train size: (553124, 78)
Test size:  (138282, 78)
All splits saved!
